In [ ]:
working_directory = "C:/Users/Kaist/Desktop/EE305_module2" # working director, change this if you use your PC
# change current directory to the working directory
%cd {working_directory}

In [ ]:
import sys
print(sys.version) # print Python version
print(sys.executable) # print path of the Python executable

In [ ]:
import paramiko # library for SSH communication
import numpy # numerical computing library
import matplotlib # library for plotting and visualization
import time # library for time-related function
import PyQt6 # library for GPU framework (for real time plotting)

print("All imports successful")

In [ ]:
HOST = "ee30005.local" # raspberry PI's hostname
USER = "student" # user name
PW   = "student" # password

ssh = paramiko.SSHClient() # create SSH client instance

ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy()) # skip host key verification

# try to establish SSH connection and execute command
try:
    ssh.connect(HOST, username=USER, password=PW, timeout=10) # connect to raspberrp PI using SSH credentials (10s timeout)
    print("SSH connection success!")

    stdin, stdout, stderr = ssh.exec_command("hostname") # execute 'hostname' command on raspberry PI
    print("Remote Hostname:", stdout.read().decode().strip())

except Exception as e:
    print("SSH connection fail:", e) # if there is an error, print log

finally:
    ssh.close() # ensure SSH connection is closed

In [ ]:
# SSH connection to raspberry PI
def ssh_connect():
    ssh = paramiko.SSHClient() # create new SSH client
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy()) # skip verification
    ssh.connect(HOST, username=USER, password=PW, timeout=10) # connect
    return ssh

# execute command in raspberry PI
def ssh_run(ssh, cmd, timeout=120):
    stdin, stdout, stderr = ssh.exec_command(cmd, timeout=timeout) # excute command, capture input/output/error streams
    out = stdout.read().decode(errors="replace") # read stdout, decode bytes to string (replace invalid chars)
    err = stderr.read().decode(errors="replace") # read stderr, decode bytes to string (replace invalid chars)
    return out, err

In [ ]:
cmd = f"arecord -l" # command to list available audio recording devices (ALSA)

ssh = ssh_connect() # ssh connection
out, err = ssh_run(ssh, cmd, timeout=10) # execute arecord -l command (10s timeout), capture output/error
print("---- arecord stdout ----")
print(out)  # print captured hardware devices list from stdout
print("---- arecord stderr ----")
print(err) # print stderr (usually empty for this command)

In [ ]:
SAMPLING_RATE = 48000 # sampling rate : 48000 Hz
CHANNEL = 1 # channel : 1
DEVICE = "plughw:0,0" # If you have a problem, change it to "default"
RECORDING_DURATION = 20 # recording duration : 20 seconds
DATATYPE = "raw" # get raw PCM data (not including .wav header)
BUFFER_SIZE = 4096 # buffer size
PERIOD_SIZE = 1024 # period size, when the buffer accumulates one period of samples, ALSA signals that the data is ready to be transferred

cmd = f"arecord -D {DEVICE} -f S32_LE -r {SAMPLING_RATE} -c {CHANNEL} -d {RECORDING_DURATION} -t {DATATYPE} --buffer-size {BUFFER_SIZE} --period-size {PERIOD_SIZE}" # recording command

In [ ]:
import numpy as np
def remove_dc_offset(x): 
    return x - np.mean(x) # Remove dc offset

# Convert frequency in Hz to the Mel scale (HTK formula)
def hz_to_mel(f): 
    return 2595.0 * np.log10(1.0 + f / 700.0) 

# Convert frequency in the Mel scale back to Hz (inverse HTK formula)
def mel_to_hz(m):
    return 700.0 * (10.0**(m / 2595.0) - 1.0)

# Compute frequencies (in Hz) evenly spaced on the Mel scale
def mel_frequencies(n_mels, fmin, fmax):
    min_mel = hz_to_mel(fmin) # minimum frequency(Hz) to mel scale
    max_mel = hz_to_mel(fmax) # maximum frequency(Hz) to mel scale
    mels = np.linspace(min_mel, max_mel, n_mels) # center frequencies in mel scale
    mels_hz = mel_to_hz(mels) # center frequencies - back to Hz
    return mels_hz

# Create a Mel filter bank 
def mel_filter(fft_size, fs, n_mels, fmin, fmax):
  weights = np.zeros((n_mels, int(1 + fft_size // 2))) # Initialize the filter bank matrix (n_mels × n_fft_bins)
  fftfreqs = np.fft.rfftfreq(n=fft_size, d=1.0 / fs) # FFT bin center frequencies (in Hz)
  mel_f = mel_frequencies(n_mels + 2, fmin, fmax) # Mel frequencies
  fdiff = np.diff(mel_f) # Frequency spacing between adjacent Mel frequencies
  ramps = np.subtract.outer(mel_f, fftfreqs) # Outer difference: each Mel edge minus all FFT bin frequencies

  # Construct each triangular Mel filter
  for i in range(n_mels):
    lower = -ramps[i] / fdiff[i] # Lower slope
    upper = ramps[i + 2] / fdiff[i + 1] # Upper slope
    weights[i] = np.maximum(0, np.minimum(lower, upper)) # Take intersection (triangle shape) and clip negative values to 0
    
  enorm = 2.0 / (mel_f[2:n_mels+2] - mel_f[:n_mels]) # Normalize each filter by its bandwidth 
  weights *= enorm[:, np.newaxis]

  return weights

def compute_single_frame_mel_spectrogram(x, chunk_size, window, fft_size, MEL_FB):
    if len(x) < chunk_size: 
        x = np.concatenate((x, np.zeros(chunk_size - len(x), dtype=x.dtype)))  # Zero-pad the signal if it is shorter than the chunk size
    xw = x * window # Applying the window
    xw = xw/np.linalg.norm(window)*np.sqrt(len(window)) # Compensate for energy loss due to windowing
    
    X = np.fft.rfft(xw, n=fft_size) # Compute the real FFT of the windowed signal
    mag = np.abs(X) / chunk_size # Scale magnitude by signal length
    power = mag**2 # power
    if fft_size % 2 == 0: # Convert to single-sided spectrum (preserve DC and Nyquist if present)  
        power[1:-1] *= 2.0 # Even FFT size → includes Nyquist bin
    else:
        power[1:] *= 2.0 # Odd FFT size → no Nyquist bin
    S_mel = MEL_FB @ power # apply mel filter to power
    mel_db = 10.0 * np.log10(S_mel) # Convert Mel-band power to dB
    return mel_db

In [ ]:
## Do not stop the cell while the code is running.
# If no audio device is found, or if only a busy audio device is found after stopping the code,
# reboot the Raspberry Pi.

# Use Qt backend for real-time plotting (separate GUI window)
%matplotlib qt
import time
import numpy as np
import matplotlib.pyplot as plt

# constants for receiving data & processing data
CHUNK_SIZE = 4096  # Number of samples per processing 
WINDOW = np.hanning(CHUNK_SIZE).astype(np.float32) # Hann window for spectral analysis
FFT_SIZE = 4096   # Number of FFT points
HOP_SIZE = CHUNK_SIZE // 2 # Hop size 

BYTES_PER_SAMPLE = 4      # 4 bytes per sample for S32_LE format
CHUNK_BYTES = CHUNK_SIZE * BYTES_PER_SAMPLE # Number of bytes per processing block
HOP_BYTES = HOP_SIZE * BYTES_PER_SAMPLE # Number of bytes for hop
RECV_SIZE = CHUNK_BYTES * 8  # Maximum number of bytes to read from the SSH channel at once
MAX_BACKLOG_CHUNKS = 10 # Maximum number of chunks allowed in the backlog
MAX_BACKLOG_BYTES = CHUNK_BYTES * MAX_BACKLOG_CHUNKS # Maximum backlog size in bytes

# Constants for Mel spectrogram
N_MELS = 128 # number of mel bands
# N_MELS = 32
FMIN = 20 # minimum frequency
FMAX = SAMPLING_RATE / 2 # maximum frequency
MEL_FB = mel_filter(FFT_SIZE, SAMPLING_RATE, N_MELS, FMIN, FMAX) # Mel filter bank

# constants for plotting
FPS = 20                # Plot update rate in frames per second
PLOT_INTERVAL = 1.0 / FPS   # Minimum time interval between plot updates
N_FRAMES = 200 # Number of time frames shown in the spectrogram
V_MAX = -60 # Upper limit for color scale (dB)
V_MIN = -150 # Lower limit for color scale (dB)

# Figure initialization
plt.ion()
mel_buffer = np.full((N_MELS, N_FRAMES), -120.0, dtype=np.float32) # Buffer to store Mel spectrogram frames [mel_bins, time_frames]

fig, ax = plt.subplots(figsize=(10, 5)) # Initialize image for Mel spectrogram
img = ax.imshow(
    mel_buffer,  # 2D data to display (Mel spectrogram buffer: mel_bins x time_frames)
    origin="lower",  # Set origin to lower-left (low → high frequency from bottom to top)
    aspect="auto",  # Automatically adjust aspect ratio to fit the figure
    extent=[0, N_FRAMES, 0, N_MELS], # 
    vmin=V_MIN,  # Lower bound of colormap (dB): values below are clipped to minimum color
    vmax=V_MAX,  # Upper bound of colormap (dB): values above are clipped to maximum color
    animated=True,  # Enable blitting optimization for real-time updates
    cmap="inferno"  # Colormap used for visualization
) 

ax.set_title(f"Real-time Mel Spectrogram (Mono, plughw), CHUNK_SIZE={CHUNK_SIZE}, FFT_SIZE={FFT_SIZE}, N_MELS={N_MELS}") # title
ax.set_xlabel("Time Frame") # X-axis is frame index (not time in seconds)
ax.set_ylabel("Mel Bin")  # Y-axis is Mel filter index (

cbar = fig.colorbar(img, ax=ax)  # Colorbar for magnitude (dB scale)
cbar.set_label("dB")

fig.canvas.draw() # Draw once to initialize canvas
background = fig.canvas.copy_from_bbox(ax.bbox) # Save static background for fast blitting updates
ax.draw_artist(img) # Draw initial image
fig.canvas.blit(ax.bbox)
fig.canvas.flush_events()

# SSH connection
ssh = ssh_connect() # Draw initial image
stdin, stdout, stderr = ssh.exec_command(cmd) # Establish SSH connection
channel = stdout.channel # Execute arecord command remotely

# Buffers and counters
buffer_pcm = bytearray() # Buffer for incoming PCM bytes
chunk_count = 0 # Number of dropped chunks due to backlog overflow
dropped_chunks_count = 0 # Number of dropped chunks due to backlog overflow

# Real-time mel spectrogram loop
time.sleep(0.1) # wait for plot initialization
print("Starting: real-time mel spectrogram...")
start_time = time.time() # start time
last_plot_time = 0.0 # last plot time

while True:
    # ** Stop slightly before the nominal recording duration to avoid potential GUI (figure window) issue **
    if (time.time() - start_time) > (RECORDING_DURATION - 0.2):  # ** higly recommended **
        break

    if channel.recv_ready():
        data = channel.recv(RECV_SIZE) # Read up to RECV_SIZE(maximum) bytes from the SSH channel
        buffer_pcm.extend(data) # Append received bytes to the local PCM buffer

        # If the backlog grows too large, drop the oldest chunks (to keep the display close to real time)
        if len(buffer_pcm) > MAX_BACKLOG_BYTES:
            overflow_bytes = len(buffer_pcm) - MAX_BACKLOG_BYTES
            drop_chunks = int(np.ceil(overflow_bytes / CHUNK_BYTES)) # Convert overflow size from bytes to whole-chunk units
            drop_bytes = drop_chunks * CHUNK_BYTES

            del buffer_pcm[:drop_bytes] # Remove the oldest buffered bytes
            dropped_chunks_count += drop_chunks # Count the dropped chunks
            print(f'{drop_chunks}chunks dropped!') # Print the number of dropped chunks in this loop

        # Process chunk in the buffer
        while len(buffer_pcm) >= CHUNK_BYTES: 
            pcm_bytes = buffer_pcm[:CHUNK_BYTES] # Extract one chunk of PCM bytes

            x = np.frombuffer(pcm_bytes, dtype="<i4").astype(np.float32) # Convert raw little-endian int32 PCM bytes to float32 samples
            x /= (2**31) # Normalize int32 PCM values to [-1, 1]

            x = remove_dc_offset(x) # Remove DC offset before spectral analysis

            mel_db = compute_single_frame_mel_spectrogram(     # Compute Mel spectrogram frame (in dB)
                x, CHUNK_SIZE, WINDOW, FFT_SIZE, MEL_FB
            )

            mel_buffer[:, :-1] = mel_buffer[:, 1:] # Shift left and append newest mel frame
            mel_buffer[:, -1] = mel_db

            del buffer_pcm[:HOP_BYTES] # Remove the extracted chunk from the buffer
            chunk_count += 1 # Increment the total chunk count

            now = time.time()
            if now - last_plot_time > PLOT_INTERVAL:  # Shift left and append newest mel frame
                img.set_data(mel_buffer) # Update the mel specrogram data
                fig.canvas.restore_region(background) # Restore the saved background
                ax.draw_artist(img) # Draw the line artist on top of the background
                fig.canvas.blit(ax.bbox) # Update only the axes region for efficient real-time rendering
                fig.canvas.flush_events() # Process GUI events so the initial frame appears

                last_plot_time = now # Update the last plot time

            if chunk_count % 100 == 0:
                print(f"chunk: {chunk_count}") # Print progress every 100 processed chunks

    elif channel.exit_status_ready():
        # Exit the loop when the remote process has finished and there is no more data left to read
        if not channel.recv_ready():
            break
    else:
        time.sleep(0.001) # Small sleep to avoid busy-waiting when no data is available

# Cleanup
err = stderr.read().decode("utf-8", errors="ignore") # Read stderr from the remote process
ssh.close() # Close SSH connection

print("---- arecord stderr ----")
print(err)
print(f"number of total chunks: {chunk_count}")
print(f"number of dropped chunks: {dropped_chunks_count}")

plt.ioff()